In [ ]:
def skip_lines(file, linesToSkip):
    for i in range(0, linesToSkip):
        file.readline()

## Species_Info

In [47]:
import pandas as pd
TOTAL_POKEMON = 386

def read_species_info_data(pokedex):
    print(f"{"Reading data from species_info.h":40}", end="")
    with open("./../src/data/pokemon/species_info.h", "r") as file:
        skip_lines(file, 33)
        for i in range(0, TOTAL_POKEMON):
            pkmn = dict()
            pkmn["index"] = i + 1
            pkmn["codeName"] = file.readline().strip()[9:-3]
            pkmn["gameName"] = ""
            skip_lines(file, 1)
            pkmn["HP"] = int(file.readline().strip()[10:-1])
            pkmn["phAtk"] = int(file.readline().strip()[14:-1])
            pkmn["phDef"] = int(file.readline().strip()[15:-1])
            pkmn["speed"] = int(file.readline().strip()[13:-1])
            pkmn["spAtk"] = int(file.readline().strip()[16:-1])
            pkmn["spDef"] = int(file.readline().strip()[17:-1])
            pkmn["BST"] = pkmn["HP"] + pkmn["phAtk"] + pkmn["phDef"] + pkmn["speed"] + pkmn["spAtk"] + pkmn["spDef"]
            [pkmn["type1"], pkmn["type2"]] = file.readline().strip()[10:-2].split(", ")
            pkmn["catchRate"] = int(file.readline().strip()[13:-1])
            pkmn["expYield"] = int(file.readline().strip()[12:-1])
            pkmn["evYield_HP"] = int(file.readline().strip()[14:-1])
            pkmn["evYield_phAtk"] = int(file.readline().strip()[18:-1])
            pkmn["evYield_phDef"] = int(file.readline().strip()[19:-1])
            pkmn["evYield_speed"] = int(file.readline().strip()[17:-1])
            pkmn["evYield_spAtk"] = int(file.readline().strip()[20:-1])
            pkmn["evYield_spDef"] = file.readline().strip()[21:-1]
            pkmn["itemCommon"] = file.readline().strip()[14:-1]
            pkmn["itemRare"] = file.readline().strip()[12:-1]
            pkmn["genderRatio"] = file.readline().strip()[15:-1]
            pkmn["eggCycles"] = int(file.readline().strip()[13:-1])
            pkmn["friendship"] = int(file.readline().strip()[14:-1])
            pkmn["growthRate"] = file.readline().strip()[14:-1]
            [pkmn["eggGroup1"], pkmn["eggGroup2"]] = file.readline().strip()[14:-2].split(", ")
            [pkmn["ability1"], pkmn["ability2"]] = file.readline().strip()[14:-2].split(", ")
            pkmn["safariZoneFleeRate"] = int(file.readline().strip()[22:-1])
            pkmn["bodyColor"] = file.readline().strip()[13:-1]
            pkmn["noFlip"] = file.readline().strip()[10:-1]
            skip_lines(file, 26 if (i == 250) else 1)
            pokedex.append(pkmn)
    print(": complete")
    print(f"{"Reading data from species_names.h":40}", end="")
    with open("./../src/data/text/species_names.h", "r") as file:
        skip_lines(file, 2)
        for i in range(0, TOTAL_POKEMON):
            offset = 16 + len(pokedex[i]["codeName"])
            pokedex[i]["gameName"] = file.readline().strip()[offset:-3]
    print(": complete")

def write_pokedex_text_data(pokedex):
    print(f"{"Writing data in pokedex_text_fr.h":40}", end="")
    with open("./../src/data/pokemon/pokedex_text_fr.h", "w", encoding="utf-8") as file:
        file.write("const u8 gDummyPokedexText[] = _(\n")
        file.write("    \"This is a newly discovered POKéMON. It is\\n\"\n")
        file.write("    \"currently under investigation. No detailed\\n\"\n")
        file.write("    \"information is available at this time.\");\n")
        file.write("\nconst u8 gDummyPokedexTextUnused[] = _(\"\");\n")

        for i in range(TOTAL_POKEMON):
            name = pokedex[i]["codeName"]
            exceptions = {"NIDORAN_F": "NidoranF", "NIDORAN_M": "NidoranM",
                          "MR_MIME": "Mrmime", "HO_OH": "HoOh"}
            if name in exceptions:
                name = exceptions[name]
            else:
                name = name.title()
            file.write(f"\nconst u8 g{name}PokedexText[] = _(\n")
            file.write(f"    \"HP     - {pokedex[i]["HP"]:3}    Ph.Atk - {pokedex[i]["phAtk"]:3}\\n\"\n")
            file.write(f"    \"Ph.Def - {pokedex[i]["phDef"]:3}    Sp.Atk - {pokedex[i]["spAtk"]:3}\\n\"\n")
            file.write(f"    \"Sp.Def - {pokedex[i]["spDef"]:3}    Speed  - {pokedex[i]["speed"]:3}\");\n")
            file.write(f"\nconst u8 g{name}PokedexTextUnused[] = _(\"\");\n")
    print(": complete")

def read_pokedex_order(pokedex):
    print(f"{"Reading data from pokedex.h":40}", end="")
    with open("./../include/constants/pokedex.h", "r") as file:
        skip_lines(file, 7)
        newPokedex = list()
        for i in range(TOTAL_POKEMON):
            name = file.readline().strip()[13:-1]
            matchFound = False
            for j in range(TOTAL_POKEMON):
                if(pokedex[j]["codeName"] == name):
                    newPokedex.append(pokedex[j])
                    matchFound = True
                    break
            if(not matchFound):
                print(f"\nFailed to match {name}")
        if(len(pokedex) != len(newPokedex)):
            print(f"Failed to match {len(pokedex) - len(newPokedex)} pokemon")
            return
        else:
            pokedex = newPokedex
            for i in range(TOTAL_POKEMON):
                pokedex[i]["index"] = int(i + 1)
    print(": complete")
    print(f"{"Writing data in new_pokedex.js":40}", end="")
    with open("./webpage/new_pokedex.js", "w", encoding="utf-8") as file:
        file.write("const pokemon_names = [\n")
        for i in range(0, TOTAL_POKEMON):
            if (pokedex[i]["codeName"] == "UNOWN"):
                file.write("\t\"UNOWN/A\",\n")
            elif (pokedex[i]["codeName"] == "CASTFORM"):
                file.write("\t\"CASTFORM/NORMAL\",\n")
            else:
                file.write(f"\t\"{pokedex[i]["codeName"]}\",\n")
        file.write("];\n\n")
    print(": complete")
    print(f"{"Writing data in pokedex.csv":40}", end="")
    df = pd.DataFrame(pokedex)
    df.to_csv("./datasets/species_info.csv", index=False)
    print(": complete")
    return df

# __main__ ()
pokedex = list()
read_species_info_data(pokedex)
write_pokedex_text_data(pokedex)
read_pokedex_order(pokedex)
print("Success!!")

Reading data from species_info.h        : complete
Reading data from species_names.h       : complete
Writing data in pokedex_text_fr.h       : complete
Reading data from pokedex.h             : complete
Writing data in new_pokedex.js          : complete
Writing data in pokedex.csv             : complete
Success!!


## Evolutions

In [71]:
import pandas as pd
TOTAL_POKEMON = 386

def read_evolutions_data(evolutions):
    print(f"{"Reading evolution data from evolution.h":40}", end="")
    with open("./../src/data/pokemon/evolution.h", "r") as file:
        skip_lines(file, 2)
        base_species = None
        branch_evo = False
        for line in file:
            evo = dict()
            line = line.strip()
            if(line[0] == "}"):
                break
            if(not branch_evo):
                [LHS, RHS] = line.split("=")
                base_species = LHS.strip()[9:-1]
                RHS = RHS.strip()[:-1]
            else:
                RHS = line.strip()[:-1]

            evo["baseSpecies"] = base_species
            (start, stop) = (2, -2)
            branch_evo = False
            if(RHS[1] != "{"):
                start = 1
            if(RHS[-2] != "}"):
                stop = -2
                branch_evo = True
            RHS = RHS[start:stop]
            [evo["method"], evo["param"], evo["targetSpecies"]] = RHS.split(", ")
            evo["targetSpecies"] = evo["targetSpecies"][8:]
            evolutions.append(evo)
    df = pd.DataFrame(evolutions)
    df.to_csv("./datasets/evolutions.csv", index=False)
    print(": complete")
    # return evolutions

def write_evolutions_data(evolutions):
    pass

# __main__ ()
evolutions = list()
read_evolutions_data(evolutions)
print("Success!!")

Reading evolution data from evolution.h : complete
Success!!


## Moves

## Movesets